# Semantic Model Similarity - Results

Interactive app for the semantic model similarity analysis. It reads the scored results from the lakehouse (written by the **semantic_model_similarity** notebook) and renders them below. Run the notebook top to bottom and explore - there is no code to edit.

In [ ]:
def render_results():
    """Load the latest results from the lakehouse and render the interactive app."""
    import hashlib
    import json
    import re
    from datetime import datetime, timezone

    import pandas as pd
    from pyspark.sql.utils import AnalysisException

    def load_delta(table_name):
        return spark.read.format("delta").load("Tables/" + table_name).toPandas()

    models_df = load_delta("semantic_models")
    tables_df = load_delta("semantic_model_tables")
    columns_df = load_delta("semantic_model_columns")
    relationships_df = load_delta("semantic_model_relationships")
    measures_df = load_delta("semantic_model_measures")
    datasources_df = load_delta("semantic_model_datasources")
    pairs_df = load_delta("semantic_model_similarity_pairs")
    try:
        run_meta_df = load_delta("semantic_model_similarity_run")
    except AnalysisException:
        run_meta_df = pd.DataFrame()

    if models_df.empty:
        raise ValueError(
            "No rows in semantic_models in the attached lakehouse. "
            "Run semantic_model_tom_catalog first."
        )

    def norm(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return re.sub(r"\s+", " ", str(value)).strip().casefold()

    def norm_dax(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        text = re.sub(r"/\*.*?\*/", " ", str(value), flags=re.DOTALL)
        text = re.sub(r"//.*", " ", text)
        return re.sub(r"\s+", " ", text).strip().casefold()

    def disp(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return str(value)

    def rounded(value):
        try:
            return round(float(value), 4)
        except (TypeError, ValueError):
            return None

    signatures = {}
    for _, row in models_df.iterrows():
        model_id = str(row["model_id"])
        signatures[model_id] = {
            "model_id": model_id,
            "workspace_id": disp(row.get("workspace_id")),
            "workspace_name": disp(row.get("workspace_name")),
            "model_name": disp(row.get("model_name")) or model_id,
        }

    if not run_meta_df.empty:
        meta = run_meta_df.iloc[0]
        duplicate_threshold = float(meta["duplicate_threshold"])
        similar_threshold = float(meta["similar_threshold"])
        containment_threshold = float(meta["containment_threshold"])
    else:
        duplicate_threshold, similar_threshold, containment_threshold = 0.95, 0.70, 0.95

    # Shared DAX pool: repeated expressions are serialized once.
    dax_pool = []
    dax_index = {}

    def dax_id(text):
        if not text:
            return -1
        idx = dax_index.get(text)
        if idx is None:
            idx = len(dax_pool)
            dax_pool.append(text)
            dax_index[text] = idx
        return idx

    def dax_hash(text):
        normalized = norm_dax(text)
        return hashlib.md5(normalized.encode("utf-8")).hexdigest()[:12] if normalized else ""

    # Inventories include every catalog model so direct Compare and the map cover the full comparable estate.
    inventories = {
        model_id: {
            "tables": {},
            "columns": {},
            "measures": {},
            "relationships": {},
            "datasources": {},
        }
        for model_id in signatures
    }

    for _, row in tables_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = norm(row["table_name"])
            if key:
                inventories[model_id]["tables"].setdefault(key, disp(row["table_name"]))

    for _, row in columns_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            table_key, column_key = norm(row["table_name"]), norm(row["column_name"])
            key = table_key + "." + column_key
            if column_key:
                inventories[model_id]["columns"].setdefault(
                    key,
                    {
                        "table": disp(row["table_name"]),
                        "tableKey": table_key,
                        "name": disp(row["column_name"]),
                    },
                )

    for _, row in measures_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = norm(row["measure_name"])
            expression = disp(row.get("expression"))
            if key:
                inventories[model_id]["measures"].setdefault(
                    key,
                    {
                        "name": disp(row["measure_name"]),
                        "daxId": dax_id(expression),
                        "daxHash": dax_hash(expression),
                    },
                )

    for _, row in relationships_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = (
                f"{norm(row['from_table'])}.{norm(row['from_column'])}"
                f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
            )
            inventories[model_id]["relationships"].setdefault(
                key,
                {
                    "from": f"{disp(row['from_table'])}[{disp(row['from_column'])}]",
                    "to": f"{disp(row['to_table'])}[{disp(row['to_column'])}]",
                },
            )

    for _, row in datasources_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            connection = (
                row.get("connection_string")
                or row.get("connection_details")
                or row.get("datasource_name")
            )
            key = norm(connection)
            if key:
                inventories[model_id]["datasources"].setdefault(key, disp(connection))

    models_payload = {}
    for model_id, identity in signatures.items():
        inv = inventories[model_id]
        models_payload[model_id] = {
            "name": identity["model_name"],
            "workspace": identity["workspace_name"],
            "tables": [{"key": k, "name": v} for k, v in inv["tables"].items()],
            "columns": [
                {
                    "key": k,
                    "table": v["table"],
                    "tableKey": v["tableKey"],
                    "name": v["name"],
                }
                for k, v in inv["columns"].items()
            ],
            "measures": [
                {
                    "key": k,
                    "name": v["name"],
                    "daxId": v["daxId"],
                    "daxHash": v["daxHash"],
                }
                for k, v in inv["measures"].items()
            ],
            "relationships": [
                {"key": k, "from": v["from"], "to": v["to"]}
                for k, v in inv["relationships"].items()
            ],
            "datasources": [{"key": k, "name": v} for k, v in inv["datasources"].items()],
        }

    def pair_object(row):
        id_a, id_b = str(row["model_id_a"]), str(row["model_id_b"])
        sig_a, sig_b = signatures.get(id_a, {}), signatures.get(id_b, {})
        coverage_a_in_b = rounded(row.get("model_a_in_model_b"))
        coverage_b_in_a = rounded(row.get("model_b_in_model_a"))
        relationship = disp(row.get("containment_relationship"))

        contained_id = containing_id = ""
        contained_coverage = None
        if relationship == "model_a_contains_model_b":
            contained_id, containing_id, contained_coverage = id_b, id_a, coverage_b_in_a
        elif relationship == "model_b_contains_model_a":
            contained_id, containing_id, contained_coverage = id_a, id_b, coverage_a_in_b
        elif relationship == "equivalent":
            contained_coverage = max(coverage_a_in_b or 0, coverage_b_in_a or 0)
        elif (coverage_a_in_b or 0) >= (coverage_b_in_a or 0):
            contained_id, containing_id, contained_coverage = id_a, id_b, coverage_a_in_b
        else:
            contained_id, containing_id, contained_coverage = id_b, id_a, coverage_b_in_a

        return {
            "idA": id_a,
            "idB": id_b,
            "modelA": sig_a.get("model_name") or disp(row.get("model_a")) or id_a,
            "workspaceA": sig_a.get("workspace_name") or disp(row.get("workspace_a")),
            "modelB": sig_b.get("model_name") or disp(row.get("model_b")) or id_b,
            "workspaceB": sig_b.get("workspace_name") or disp(row.get("workspace_b")),
            "scored": True,
            "composite": rounded(row.get("composite_score")),
            "containment": rounded(row.get("containment_score")),
            "relationship": relationship,
            "containedId": contained_id,
            "containingId": containing_id,
            "containedCoverage": contained_coverage,
            "aInB": coverage_a_in_b,
            "bInA": coverage_b_in_a,
            "crossWorkspace": bool(row.get("cross_workspace", False)),
            "sameName": bool(row.get("same_model_name", False)),
            "jaccard": {
                "tables": rounded(row.get("jaccard_tables")),
                "columns": rounded(row.get("jaccard_columns")),
                "measures": rounded(row.get("jaccard_measure_names")),
                "relationships": rounded(row.get("jaccard_relationships")),
                "datasources": rounded(row.get("jaccard_datasources")),
            },
            "daxCosine": rounded(row.get("dax_embedding_cosine")),
        }

    all_pairs_payload = [pair_object(row) for _, row in pairs_df.iterrows()]
    model_list = sorted(
        [
            {
                "id": model_id,
                "name": model["name"],
                "workspace": model["workspace"],
            }
            for model_id, model in models_payload.items()
        ],
        key=lambda item: (item["name"].casefold(), item["workspace"].casefold()),
    )

    default_a = default_b = ""
    if all_pairs_payload:
        best_pair = max(all_pairs_payload, key=lambda p: p["composite"] or 0)
        default_a, default_b = best_pair["idA"], best_pair["idB"]
    elif len(model_list) >= 2:
        default_a, default_b = model_list[0]["id"], model_list[1]["id"]

    app_data = {
        "generatedAt": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        "summary": {"models": len(model_list)},
        "thresholds": {
            "duplicate": rounded(duplicate_threshold),
            "similar": rounded(similar_threshold),
            "containment": rounded(containment_threshold),
        },
        "allPairs": all_pairs_payload,
        "models": models_payload,
        "modelList": model_list,
        "daxPool": dax_pool,
        "defaultCompare": {"a": default_a, "b": default_b},
    }

    app_template = r"""
<script>
  (() => {
    const param = new URLSearchParams(window.location.search).get("scoutTheme");
    const theme =
      param || (window.matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light");
    document.documentElement.setAttribute("data-theme", theme);
  })();
</script>
<style>
:root {
  color-scheme: light;
  --cp-bg: #f7f4ef;
  --cp-bg-elevated: #fcfbf8;
  --cp-surface: #ffffff;
  --cp-surface-soft: #f5f5f5;
  --cp-border: #dedede;
  --cp-border-strong: #919191;
  --cp-text: #242424;
  --cp-text-muted: #5c5c5c;
  --cp-text-soft: #6f6f6f;
  --cp-accent: #b11f4b;
  --cp-accent-hover: #9a1a41;
  --cp-accent-soft: rgba(177, 31, 75, 0.08);
  --cp-accent-fg: #ffffff;
  --cp-success: #16a34a;
  --cp-danger: #dc2626;
  --cp-warning: #f59e0b;
  --cp-link: #0078d4;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.12);
  --cp-overlay: rgba(255, 255, 255, 0.8);
  --cp-panel: rgba(255, 255, 255, 0.86);
  --cp-panel-strong: rgba(255, 255, 255, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.55);
  --cp-highlight: rgba(177, 31, 75, 0.12);
}
html[data-theme="dark"] {
  color-scheme: dark;
  --cp-bg: #3d3b3a;
  --cp-bg-elevated: #343231;
  --cp-surface: #292929;
  --cp-surface-soft: #2e2e2e;
  --cp-border: #474747;
  --cp-border-strong: #5f5f5f;
  --cp-text: #dedede;
  --cp-text-muted: #919191;
  --cp-text-soft: #b0b0b0;
  --cp-accent: #fd8ea1;
  --cp-accent-hover: #fb7b91;
  --cp-accent-soft: rgba(253, 142, 161, 0.14);
  --cp-accent-fg: #1a1a1a;
  --cp-success: #4ade80;
  --cp-danger: #f87171;
  --cp-warning: #fbbf24;
  --cp-link: #4da6ff;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.32);
  --cp-overlay: rgba(41, 41, 41, 0.88);
  --cp-panel: rgba(41, 41, 41, 0.72);
  --cp-panel-strong: rgba(41, 41, 41, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.04);
  --cp-highlight: rgba(253, 142, 161, 0.12);
}
#sms-app {
  background: var(--cp-bg);
  color: var(--cp-text);
  font-family: "Segoe UI", Aptos, Calibri, -apple-system, BlinkMacSystemFont, sans-serif;
  max-width: 1180px;
  margin: 8px auto;
  border: 1px solid var(--cp-border);
  border-radius: 16px;
  overflow: hidden;
}
#sms-app * { box-sizing: border-box; }
#sms-app button, #sms-app input, #sms-app select { font: inherit; }
#sms-app button, #sms-app select, #sms-app input[type="text"], #sms-app input[type="number"] { border-radius: 0.625rem; }
#sms-app button:focus-visible, #sms-app input:focus-visible, #sms-app select:focus-visible, #sms-app [tabindex]:focus-visible {
  outline: 3px solid var(--cp-accent);
  outline-offset: 2px;
}
#sms-app .app-head { display: flex; justify-content: space-between; gap: 16px; padding: 24px 28px 16px; }
#sms-app .eyebrow { color: var(--cp-text-muted); font-size: 12px; font-weight: 700; letter-spacing: .06em; text-transform: uppercase; }
#sms-app h1 { font-size: 23px; margin: 4px 0; }
#sms-app .subtitle, #sms-app .muted { color: var(--cp-text-muted); }
#sms-app .subtitle { font-size: 13px; }
#sms-app .head-controls { display: flex; flex-direction: column; align-items: flex-end; gap: 8px; }
#sms-app .control-row, #sms-app .toolbar, #sms-app .filter-row, #sms-app .compare-controls, #sms-app .legend { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
#sms-app .generated { color: var(--cp-text-muted); font-size: 11px; }
#sms-app .button, #sms-app .tab, #sms-app .filter, #sms-app .theme-button, #sms-app .disclosure {
  border: 1px solid var(--cp-border);
  background: var(--cp-surface);
  color: var(--cp-text);
  cursor: pointer;
  padding: 8px 12px;
}
#sms-app .button:hover, #sms-app .tab:hover, #sms-app .filter:hover, #sms-app .theme-button:hover, #sms-app .disclosure:hover { border-color: var(--cp-border-strong); }
#sms-app .button.primary { background: var(--cp-accent); border-color: var(--cp-accent); color: var(--cp-accent-fg); font-weight: 700; }
#sms-app .button.primary:hover { background: var(--cp-accent-hover); }
#sms-app .theme-button.active, #sms-app .filter.active { background: var(--cp-accent-soft); border-color: var(--cp-accent); color: var(--cp-accent); font-weight: 700; }
#sms-app .tabs { display: flex; gap: 4px; padding: 4px; margin: 0 28px; background: var(--cp-surface-soft); border-radius: 0.625rem; overflow-x: auto; }
#sms-app .tab { flex: 0 0 auto; border-color: var(--cp-surface-soft); background: var(--cp-surface-soft); font-weight: 700; }
#sms-app .tab[aria-selected="true"] { background: var(--cp-surface); border-color: var(--cp-border); color: var(--cp-accent); }
#sms-app .settings { margin: 8px 28px 0; }
#sms-app .settings-panel { background: var(--cp-surface); border: 1px solid var(--cp-border); border-radius: 16px; padding: 16px; box-shadow: 0 0 2px var(--cp-border), 0 1px 2px var(--cp-border); }
#sms-app .settings-grid { display: grid; gap: 12px; }
#sms-app .setting { display: grid; grid-template-columns: minmax(180px, 1fr) 2fr 84px; gap: 12px; align-items: center; }
#sms-app .setting small { display: block; color: var(--cp-text-muted); }
#sms-app input[type="range"] { width: 100%; accent-color: var(--cp-accent); }
#sms-app input[type="number"], #sms-app input[type="text"], #sms-app select { background: var(--cp-surface); border: 1px solid var(--cp-border); color: var(--cp-text); padding: 8px 10px; }
#sms-app .settings-actions { display: flex; justify-content: space-between; align-items: center; gap: 12px; margin-top: 12px; padding-top: 12px; border-top: 1px solid var(--cp-border); }
#sms-app .view { padding: 22px 28px 28px; }
#sms-app .view h2 { font-size: 19px; margin: 0 0 6px; }
#sms-app .intro { color: var(--cp-text-muted); font-size: 13px; margin: 0 0 18px; }
#sms-app .stats { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 12px; margin-bottom: 18px; }
#sms-app .card, #sms-app .stat, #sms-app .empty, #sms-app .compare-summary, #sms-app .map-wrap {
  background: var(--cp-surface);
  border: 1px solid var(--cp-border);
  border-radius: 16px;
  box-shadow: 0 0 2px var(--cp-border), 0 1px 2px var(--cp-border);
}
#sms-app .stat { padding: 16px; }
#sms-app .stat strong { display: block; font-size: 29px; }
#sms-app .stat span { color: var(--cp-text-muted); font-size: 12px; }
#sms-app .toolbar { justify-content: space-between; margin: 16px 0 12px; }
#sms-app .search { flex: 1 1 220px; min-width: 180px; }
#sms-app .filter { font-size: 12px; font-weight: 700; }
#sms-app .queue { display: grid; gap: 10px; }
#sms-app .candidate { padding: 14px 16px; }
#sms-app .candidate-main { display: grid; grid-template-columns: minmax(0, 1fr) auto; gap: 16px; align-items: center; }
#sms-app .relationship { color: var(--cp-accent); font-size: 12px; font-weight: 800; text-transform: uppercase; letter-spacing: .03em; }
#sms-app .identity-row { display: flex; align-items: stretch; gap: 10px; margin: 8px 0; }
#sms-app .identity { flex: 1 1 0; min-width: 0; }
#sms-app .model-name { font-weight: 700; overflow-wrap: anywhere; }
#sms-app .workspace { color: var(--cp-text-muted); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .relation-word { align-self: center; color: var(--cp-text-muted); font-size: 12px; font-weight: 700; }
#sms-app .plain { color: var(--cp-text-soft); font-size: 13px; }
#sms-app .metadata { color: var(--cp-text-muted); font-size: 11px; margin-top: 6px; }
#sms-app .score { text-align: right; min-width: 130px; }
#sms-app .score strong { display: block; font-size: 22px; }
#sms-app .score span { color: var(--cp-text-muted); font-size: 11px; }
#sms-app .candidate-actions { display: flex; gap: 8px; justify-content: flex-end; margin-top: 10px; }
#sms-app .evidence { border-top: 1px solid var(--cp-border); margin-top: 12px; padding-top: 12px; }
#sms-app .evidence-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(120px, 1fr)); gap: 8px; }
#sms-app .evidence-item { background: var(--cp-surface-soft); border-radius: 0.625rem; padding: 8px; font-size: 11px; }
#sms-app .evidence-item strong { display: block; font-size: 14px; }
#sms-app .empty { padding: 28px; text-align: center; color: var(--cp-text-muted); }
#sms-app .groups { display: grid; gap: 12px; }
#sms-app .group { padding: 16px; }
#sms-app .group-head { display: flex; justify-content: space-between; gap: 12px; align-items: baseline; }
#sms-app .group-head h3 { margin: 0; font-size: 16px; }
#sms-app .group-members { display: grid; gap: 4px; list-style: none; padding: 0; margin: 12px 0; }
#sms-app .group-members li { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; border-top: 1px solid var(--cp-border); padding-top: 6px; }
#sms-app .group-pickers { display: grid; grid-template-columns: 1fr 1fr auto; gap: 8px; align-items: end; }
#sms-app label { color: var(--cp-text-soft); font-size: 12px; font-weight: 600; }
#sms-app label select { display: block; width: 100%; margin-top: 4px; }
#sms-app .compare-controls { margin-bottom: 14px; }
#sms-app .compare-controls label { flex: 1 1 230px; }
#sms-app .compare-summary { padding: 16px; margin-bottom: 12px; }
#sms-app .summary-grid { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 12px; margin-top: 12px; }
#sms-app .summary-item span { display: block; color: var(--cp-text-muted); font-size: 11px; }
#sms-app .summary-item strong { font-size: 14px; }
#sms-app .diff-section { margin-bottom: 10px; }
#sms-app .section-button { width: 100%; display: flex; justify-content: space-between; align-items: center; text-align: left; background: var(--cp-surface); border: 1px solid var(--cp-border); color: var(--cp-text); border-radius: 16px; padding: 13px 16px; cursor: pointer; font-weight: 700; }
#sms-app .section-body { background: var(--cp-surface); border: 1px solid var(--cp-border); border-top: 0; padding: 12px 16px; border-radius: 0 0 16px 16px; }
#sms-app .diff-row { display: flex; gap: 8px; padding: 5px 7px; border-radius: 0.625rem; font-size: 12px; }
#sms-app .diff-row.onlyA, #sms-app .diff-row.onlyB, #sms-app .diff-row.changed { background: var(--cp-accent-soft); }
#sms-app .status { color: var(--cp-accent); min-width: 92px; font-weight: 700; }
#sms-app .dax { font-family: Consolas, "Courier New", Courier, monospace; white-space: pre-wrap; overflow-wrap: anywhere; background: var(--cp-surface-soft); border: 1px solid var(--cp-border); border-radius: 0.625rem; padding: 8px; margin: 4px 0 10px 100px; }
#sms-app .map-note { border-left: 4px solid var(--cp-warning); padding: 8px 12px; background: var(--cp-surface-soft); margin-bottom: 12px; font-size: 12px; }
#sms-app .map-wrap { overflow-x: auto; padding: 16px; }
#sms-app .matrix { border-collapse: separate; border-spacing: 3px; min-width: max-content; }
#sms-app .matrix th { max-width: 150px; color: var(--cp-text-muted); font-size: 10px; font-weight: 700; text-align: left; }
#sms-app .matrix th.col { writing-mode: vertical-rl; transform: rotate(180deg); height: 180px; text-align: right; }
#sms-app .matrix tbody tr { background: var(--cp-surface); }
#sms-app .matrix tbody tr:nth-child(even) { background: var(--cp-surface-soft); }
#sms-app .matrix tbody td, #sms-app .matrix tbody th { background: inherit; }
#sms-app .matrix-cell { width: 32px; height: 32px; padding: 0; border: 1px solid var(--cp-border); border-radius: 0.625rem; font-size: 9px; font-weight: 800; }
#sms-app button.matrix-cell { cursor: pointer; }
#sms-app button.matrix-cell:hover { border-color: var(--cp-accent); outline: 0; box-shadow: 0 0 0 2px var(--cp-surface), 0 0 0 5px var(--cp-accent); position: relative; z-index: 1; }
#sms-app button.matrix-cell:focus-visible { border-color: var(--cp-accent); outline: 0; box-shadow: 0 0 0 2px var(--cp-surface), 0 0 0 5px var(--cp-accent); position: relative; z-index: 1; }
#sms-app .matrix-cell.duplicate { background: var(--cp-accent); color: var(--cp-accent-fg); }
#sms-app .matrix-cell.high { background: var(--cp-highlight); color: var(--cp-text); }
#sms-app .matrix-cell.low { background: var(--cp-surface-soft); color: var(--cp-text-muted); }
#sms-app .matrix-cell.unscored { background: var(--cp-bg-elevated); color: var(--cp-text-muted); }
#sms-app .matrix-cell.diagonal { display: inline-flex; justify-content: center; align-items: center; background: var(--cp-border); color: var(--cp-text); }
#sms-app .legend { margin-top: 12px; font-size: 11px; color: var(--cp-text-muted); }
#sms-app .legend-key { display: inline-block; width: 12px; height: 12px; border: 1px solid var(--cp-border); border-radius: 4px; vertical-align: middle; }
#sms-app .legend-key.duplicate { background: var(--cp-accent); }
#sms-app .legend-key.high { background: var(--cp-highlight); }
#sms-app .legend-key.low { background: var(--cp-surface-soft); }
#sms-app .legend-key.unscored { background: var(--cp-bg-elevated); }
#sms-app .foot { color: var(--cp-text-muted); font-size: 11px; padding: 0 28px 22px; }
#sms-app [hidden] { display: none !important; }
@media (max-width: 720px) {
  #sms-app .app-head { flex-direction: column; padding: 20px 16px 12px; }
  #sms-app .head-controls { align-items: flex-start; }
  #sms-app .tabs, #sms-app .settings { margin-left: 16px; margin-right: 16px; }
  #sms-app .view { padding: 18px 16px 22px; }
  #sms-app .stats, #sms-app .summary-grid { grid-template-columns: 1fr; }
  #sms-app .candidate-main { grid-template-columns: 1fr; }
  #sms-app .score { text-align: left; }
  #sms-app .identity-row { flex-direction: column; }
  #sms-app .relation-word { align-self: flex-start; }
  #sms-app .setting, #sms-app .group-pickers { grid-template-columns: 1fr; }
  #sms-app .group-members li { grid-template-columns: 1fr; }
}
</style>
<div id="sms-app"></div>
<script>
(function(){
  var DATA = __APP_DATA__;
  var root = document.getElementById('sms-app');
  if(!root){ return; }
  var MODELS = DATA.models || {}, MLIST = DATA.modelList || [], DAXPOOL = DATA.daxPool || [];
  var DEFAULTS = DATA.thresholds || {duplicate:.95, similar:.70, containment:.95};
  var initialTheme = document.documentElement.getAttribute('data-theme') || 'light';
  var state = {
    tab:'review', filter:'all', search:'', theme:initialTheme, settingsOpen:false,
    thresholds:{duplicate:DEFAULTS.duplicate, similar:DEFAULTS.similar, containment:DEFAULTS.containment},
    draft:{duplicate:DEFAULTS.duplicate, similar:DEFAULTS.similar, containment:DEFAULTS.containment},
    cmpA:(DATA.defaultCompare||{}).a || '', cmpB:(DATA.defaultCompare||{}).b || '',
    cmpDiffOnly:false, openSections:{}, groupSelections:{}
  };
  try {
    var savedTheme=localStorage.getItem('sms-theme');
    if(savedTheme==='light'||savedTheme==='dark'){ state.theme=savedTheme; }
    var savedThresholds=JSON.parse(localStorage.getItem('sms-thresholds')||'null');
    if(savedThresholds){ ['duplicate','similar','containment'].forEach(function(k){ if(typeof savedThresholds[k]==='number'){ state.thresholds[k]=Math.max(0,Math.min(1,savedThresholds[k])); } }); state.draft=Object.assign({},state.thresholds); }
  } catch(e){}
  if(!state.cmpA && MLIST[0]){ state.cmpA=MLIST[0].id; }
  if(!state.cmpB && MLIST[1]){ state.cmpB=MLIST[1].id; }

  function esc(s){ return String(s==null?'':s).replace(/[&<>"']/g,function(c){ return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]; }); }
  function score(v){ return v==null?'Not scored':(Math.round(v*1000)/10).toFixed(1)+'%'; }
  function model(id){ return MODELS[id] || {name:id,workspace:''}; }
  function identity(id){ var m=model(id); return '<div class="identity"><div class="model-name">'+esc(m.name)+'</div><div class="workspace">'+esc(m.workspace||'Workspace unavailable')+'</div></div>'; }
  function pairKey(a,b){ return [String(a),String(b)].sort().join('|'); }
  function pairMap(){ var out={}; (DATA.allPairs||[]).forEach(function(p){ out[pairKey(p.idA,p.idB)]=p; }); return out; }
  function daxText(item){ return item&&item.daxId>=0 ? (DAXPOOL[item.daxId]||'') : ''; }
  function tierAt(v){ return v>=state.thresholds.duplicate?'duplicate':(v>=state.thresholds.similar?'similar':'distinct'); }

  function classify(p){
    var tier=tierAt(p.composite==null?0:p.composite);
    var coverage=p.containedCoverage==null?Math.max(p.aInB||0,p.bInA||0):p.containedCoverage;
    if(tier==='duplicate'){ return {kind:'duplicate',label:'Likely duplicate',rank:0,primary:p.composite||0}; }
    if(coverage>=state.thresholds.containment){ return {kind:'containment',label:'Subset / superset',rank:1,primary:coverage}; }
    if(tier==='similar'){ return {kind:'overlap',label:'High overlap',rank:2,primary:p.composite||0}; }
    return null;
  }
  function candidates(){
    return (DATA.allPairs||[]).map(function(p){ var c=classify(p); return c?{pair:p,category:c}:null; }).filter(Boolean).sort(function(a,b){ return a.category.rank-b.category.rank || b.category.primary-a.category.primary; });
  }
  function duplicatePairs(){ return (DATA.allPairs||[]).filter(function(p){ return tierAt(p.composite||0)==='duplicate'; }); }
  function buildGroups(){
    var dup=duplicatePairs(), parent={};
    function find(x){ if(parent[x]!==x){ parent[x]=find(parent[x]); } return parent[x]; }
    function add(x){ if(!(x in parent)){ parent[x]=x; } }
    function join(a,b){ add(a);add(b);var ra=find(a),rb=find(b);if(ra!==rb){parent[rb]=ra;} }
    dup.forEach(function(p){join(p.idA,p.idB);});
    var grouped={}; Object.keys(parent).forEach(function(id){var r=find(id);(grouped[r]=grouped[r]||[]).push(id);});
    var pmap=pairMap(), groups=[];
    Object.keys(grouped).forEach(function(key){
      var members=grouped[key]; if(members.length<2){return;}
      var strongest=null;
      for(var i=0;i<members.length;i++){for(var j=i+1;j<members.length;j++){var p=pmap[pairKey(members[i],members[j])];if(p&&tierAt(p.composite||0)==='duplicate'&&(!strongest||(p.composite||0)>(strongest.composite||0))){strongest=p;}}}
      if(!strongest){return;}
      members.sort(function(a,b){return (model(a).name||'').localeCompare(model(b).name||'')||(model(a).workspace||'').localeCompare(model(b).workspace||'');});
      groups.push({members:members,strongest:strongest});
    });
    groups.sort(function(a,b){return b.members.length-a.members.length||(b.strongest.composite||0)-(a.strongest.composite||0);});
    return groups;
  }
  function relationshipText(p,c){
    if(p.relationship==='equivalent'){ return model(p.idA).name+' is equivalent to '+model(p.idB).name+' at the current containment threshold.'; }
    if(c.kind==='duplicate'){ return 'These models are structurally near-identical and should be reviewed for consolidation.'; }
    if(c.kind==='containment'){
      var contained=model(p.containedId), containing=model(p.containingId);
      return contained.name+' is contained in '+containing.name+' at '+score(p.containedCoverage)+' coverage.';
    }
    return 'These models share substantial structure or DAX logic but do not meet the duplicate threshold.';
  }
  function relevantSection(p){
    var values=[['tables',p.jaccard.tables],['measures',Math.min(p.jaccard.measures==null?1:p.jaccard.measures,p.daxCosine==null?1:p.daxCosine)],['relationships',p.jaccard.relationships],['datasources',p.jaccard.datasources]];
    values.sort(function(a,b){return (a[1]==null?1:a[1])-(b[1]==null?1:b[1]);}); return values[0][0];
  }

  function tabsHTML(){
    return '<div class="tabs" role="tablist" aria-label="Similarity results views">'+[
      ['review','Review'],['groups','Groups'],['map','Similarity map'],['compare','Compare']
    ].map(function(t){var selected=state.tab===t[0];return '<button class="tab" role="tab" id="tab-'+t[0]+'" aria-selected="'+selected+'" aria-controls="panel-'+t[0]+'" tabindex="'+(selected?'0':'-1')+'" data-tab="'+t[0]+'">'+t[1]+'</button>';}).join('')+'</div>';
  }
  function headerHTML(){
    return '<div class="app-head"><div><div class="eyebrow">Semantic Model Similarity</div><h1>Consolidation review</h1><div class="subtitle">Review likely duplicates, subset relationships, and high-overlap models.</div></div><div class="head-controls"><div class="control-row"><button class="button" data-settings aria-expanded="'+state.settingsOpen+'" aria-controls="scoring-settings">Scoring settings</button><span class="muted">Theme:</span><button class="theme-button'+(state.theme==='light'?' active':'')+'" data-theme-set="light" aria-pressed="'+(state.theme==='light')+'">Light</button><button class="theme-button'+(state.theme==='dark'?' active':'')+'" data-theme-set="dark" aria-pressed="'+(state.theme==='dark')+'">Dark</button></div><div class="generated">Generated '+esc(DATA.generatedAt)+'</div></div></div>';
  }
  function settingsHTML(){
    function row(key,label,hint){var v=state.draft[key];return '<div class="setting"><label for="range-'+key+'">'+label+'<small>'+hint+'</small></label><input id="range-'+key+'" type="range" min="0" max="1" step="0.01" value="'+v+'" data-draft-range="'+key+'" aria-label="'+label+' threshold"><input type="number" min="0" max="1" step="0.01" value="'+v.toFixed(2)+'" data-draft-number="'+key+'" aria-label="'+label+' threshold value"></div>';}
    return '<div class="settings" id="scoring-settings"'+(state.settingsOpen?'':' hidden')+'><div class="settings-panel"><div class="settings-grid">'+row('duplicate','Likely duplicate','Composite score at or above this value')+row('similar','High overlap','Composite score at or above this value')+row('containment','Subset / superset','Directional coverage at or above this value')+'</div><div class="settings-actions"><span class="muted">Thresholds reclassify existing scored pairs; they do not rescore blocked pairs.</span><div class="control-row"><button class="button" data-reset>Reset defaults</button><button class="button primary" data-apply>Apply thresholds</button></div></div></div></div>';
  }

  function evidenceHTML(p){
    var items=[['Tables',p.jaccard.tables],['Columns',p.jaccard.columns],['Measures',p.jaccard.measures],['DAX text',p.daxCosine],['Relationships',p.jaccard.relationships],['Data sources',p.jaccard.datasources],['Containment',p.containment]];
    return '<div class="evidence-grid">'+items.map(function(x){return '<div class="evidence-item"><span>'+x[0]+'</span><strong>'+score(x[1])+'</strong></div>';}).join('')+'</div>';
  }
  function candidateHTML(item,index){
    var p=item.pair,c=item.category,meta=[]; if(p.crossWorkspace){meta.push('Cross-workspace');}if(p.sameName){meta.push('Same model name');}meta.push('Composite '+score(p.composite));meta.push('Containment '+score(p.containment));
    var leftId=p.idA,rightId=p.idB,relation='compared with';
    if(p.relationship==='equivalent'){relation='is equivalent to';}
    else if(c.kind==='containment'&&p.containedId&&p.containingId){leftId=p.containedId;rightId=p.containingId;relation='is contained in';}
    return '<article class="card candidate"><div class="candidate-main"><div><div class="relationship">'+c.label+'</div><div class="identity-row">'+identity(leftId)+'<div class="relation-word">'+relation+'</div>'+identity(rightId)+'</div><div class="plain">'+esc(relationshipText(p,c))+'</div><div class="metadata">'+meta.join(' · ')+'</div></div><div class="score"><strong>'+score(c.primary)+'</strong><span>'+(c.kind==='containment'?'directional coverage':'composite similarity')+'</span><div class="candidate-actions"><button class="button disclosure" data-evidence="evidence-'+index+'" aria-expanded="false" aria-controls="evidence-'+index+'">Evidence</button><button class="button primary" data-compare-a="'+esc(p.idA)+'" data-compare-b="'+esc(p.idB)+'" data-open-section="'+relevantSection(p)+'">Review comparison</button></div></div></div><div class="evidence" id="evidence-'+index+'" hidden>'+evidenceHTML(p)+'</div></article>';
  }
  function reviewQueueHTML(){
    var all=candidates(),filtered=all.filter(function(item){if(state.filter!=='all'&&item.category.kind!==state.filter){return false;}var q=state.search.trim().toLowerCase();if(!q){return true;}var p=item.pair;return [p.modelA,p.workspaceA,p.modelB,p.workspaceB].join(' ').toLowerCase().indexOf(q)>=0;});
    return filtered.length?filtered.map(candidateHTML).join(''):'<div class="empty">No candidate pairs match this filter. The view remains available so thresholds or search can be adjusted.</div>';
  }
  function reviewHTML(){
    var all=candidates(),groups=buildGroups();
    var counts={duplicate:0,containment:0,overlap:0};all.forEach(function(x){counts[x.category.kind]++;});
    return '<h2>Review candidates</h2><p class="intro">One ranked queue for every pair requiring review. Each pair appears once under its primary relationship.</p><div class="stats"><div class="stat"><strong>'+DATA.summary.models+'</strong><span>models scanned</span></div><div class="stat"><strong>'+all.length+'</strong><span>unique candidate pairs requiring review</span></div><div class="stat"><strong>'+groups.length+'</strong><span>duplicate groups (connected components)</span></div></div><div class="toolbar"><input class="search" type="text" aria-label="Search review candidates" placeholder="Search models or workspaces" value="'+esc(state.search)+'"><div class="filter-row" aria-label="Candidate filters"><button class="filter'+(state.filter==='all'?' active':'')+'" data-filter="all">All '+all.length+'</button><button class="filter'+(state.filter==='duplicate'?' active':'')+'" data-filter="duplicate">Likely duplicate '+counts.duplicate+'</button><button class="filter'+(state.filter==='containment'?' active':'')+'" data-filter="containment">Subset / superset '+counts.containment+'</button><button class="filter'+(state.filter==='overlap'?' active':'')+'" data-filter="overlap">High overlap '+counts.overlap+'</button></div></div><div class="queue">'+reviewQueueHTML()+'</div>';
  }

  function groupOptions(members,selected){return members.map(function(id){var m=model(id);return '<option value="'+esc(id)+'"'+(id===selected?' selected':'')+'>'+esc(m.name)+' — '+esc(m.workspace)+'</option>';}).join('');}
  function groupsHTML(){
    var groups=buildGroups();
    if(!groups.length){return '<h2>Groups</h2><p class="intro">Groups are connected components of likely-duplicate pairs. They are review aids, not approved consolidations.</p><div class="empty">No duplicate groups exist at the current threshold.</div>';}
    return '<h2>Groups</h2><p class="intro">Each group is a connected component: every member connects through at least one likely-duplicate pair, but not every pair is necessarily mutually identical. Review before consolidating.</p><div class="groups">'+groups.map(function(g,i){
      var key='g'+i,sel=state.groupSelections[key]||{a:g.strongest.idA,b:g.strongest.idB};state.groupSelections[key]=sel;var representative=model(g.strongest.idA).name;
      return '<article class="card group"><div class="group-head"><h3>'+esc(representative)+' duplicate group</h3><span class="muted">Group '+(i+1)+' · '+g.members.length+' models</span></div><p class="plain">Strongest directly scored pair: '+esc(model(g.strongest.idA).name)+' and '+esc(model(g.strongest.idB).name)+' ('+score(g.strongest.composite)+').</p><ul class="group-members">'+g.members.map(function(id){var m=model(id);return '<li><span class="model-name">'+esc(m.name)+'</span><span class="workspace">'+esc(m.workspace)+'</span></li>';}).join('')+'</ul><div class="group-pickers"><label>First member<select data-group="'+key+'" data-group-side="a">'+groupOptions(g.members,sel.a)+'</select></label><label>Second member<select data-group="'+key+'" data-group-side="b">'+groupOptions(g.members,sel.b)+'</select></label><button class="button primary" data-group-compare="'+key+'">Compare selected members</button></div></article>';
    }).join('')+'</div>';
  }

  function indexBy(list){var out={};(list||[]).forEach(function(x){out[x.key]=x;});return out;}
  function unionKeys(a,b){var out={};Object.keys(a).forEach(function(k){out[k]=1;});Object.keys(b).forEach(function(k){out[k]=1;});return Object.keys(out).sort();}
  function statusOf(key,a,b){return a[key]&&b[key]?'shared':(a[key]?'onlyA':'onlyB');}
  function statusLabel(st,aName,bName){return st==='onlyA'?'Only in '+aName:(st==='onlyB'?'Only in '+bName:(st==='changed'?'Different DAX':'Shared'));}
  function sectionHTML(id,title,summary,rows){var open=!!state.openSections[id];return '<section class="diff-section"><button class="section-button" data-section="'+id+'" aria-expanded="'+open+'" aria-controls="section-'+id+'"><span>'+title+'</span><span class="muted">'+summary+'</span></button><div class="section-body" id="section-'+id+'"'+(open?'':' hidden')+'>'+((rows&&rows.length)?rows.join(''):'<div class="muted">Nothing to show.</div>')+'</div></section>';}
  function rowHTML(st,text,aName,bName,detail){return '<div class="diff-row '+st+'"><span class="status">'+esc(statusLabel(st,aName,bName))+'</span><span>'+text+'</span></div>'+(detail||'');}
  function compareData(a,b){
    var result={sections:{},shared:0,differences:0};
    var ta=indexBy(a.tables),tb=indexBy(b.tables),ca=indexBy(a.columns),cb=indexBy(b.columns),ma=indexBy(a.measures),mb=indexBy(b.measures),ra=indexBy(a.relationships),rb=indexBy(b.relationships),da=indexBy(a.datasources),db=indexBy(b.datasources);
    function simple(id,left,right,format){var rows=[],shared=0,diff=0;unionKeys(left,right).forEach(function(k){var st=statusOf(k,left,right);if(st==='shared'){shared++;}else{diff++;}if(!state.cmpDiffOnly||st!=='shared'){rows.push(rowHTML(st,format(left[k]||right[k]),a.name,b.name));}});result.sections[id]={rows:rows,shared:shared,diff:diff};result.shared+=shared;result.differences+=diff;}
    simple('tables',ta,tb,function(x){return esc(x.name);});
    simple('columns',ca,cb,function(x){return esc(x.table)+'['+esc(x.name)+']';});
    var measureRows=[],measureShared=0,measureDiff=0;unionKeys(ma,mb).forEach(function(k){var ia=ma[k],ib=mb[k],st;if(ia&&ib){st=ia.daxHash===ib.daxHash?'shared':'changed';}else{st=ia?'onlyA':'onlyB';}if(st==='shared'){measureShared++;}else{measureDiff++;}if(!state.cmpDiffOnly||st!=='shared'){var item=ia||ib,detail='';if(st==='changed'){detail='<div class="dax"><strong>'+esc(a.name)+'</strong>\n'+esc(daxText(ia))+'\n\n<strong>'+esc(b.name)+'</strong>\n'+esc(daxText(ib))+'</div>';}else if(st!=='shared'){detail='<div class="dax">'+esc(daxText(item))+'</div>';}measureRows.push(rowHTML(st,esc(item.name),a.name,b.name,detail));}});result.sections.measures={rows:measureRows,shared:measureShared,diff:measureDiff};result.shared+=measureShared;result.differences+=measureDiff;
    simple('relationships',ra,rb,function(x){return esc(x.from)+' → '+esc(x.to);});
    simple('datasources',da,db,function(x){return esc(x.name);});
    return result;
  }
  function relationshipSummary(p){if(!p){return 'Pair was not scored';}var c=classify(p);return c?c.label:'Low overlap';}
  function containmentSummary(p){if(!p){return 'Not scored';}if(p.relationship==='equivalent'){return 'Equivalent coverage';}if(p.containedId&&p.containingId){return model(p.containedId).name+' is contained in '+model(p.containingId).name+' ('+score(p.containedCoverage)+')';}return 'No threshold-level direction';}
  function selectOptions(selected){return MLIST.map(function(m){return '<option value="'+esc(m.id)+'"'+(m.id===selected?' selected':'')+'>'+esc(m.name)+' — '+esc(m.workspace)+'</option>';}).join('');}
  function compareHTML(){
    if(MLIST.length<2){return '<h2>Compare</h2><p class="intro">Inspect shared and different model objects.</p><div class="empty">At least two catalog models are required.</div>';}
    var a=model(state.cmpA),b=model(state.cmpB),p=pairMap()[pairKey(state.cmpA,state.cmpB)];
    if(state.cmpA===state.cmpB){return '<h2>Compare</h2><div class="compare-controls"><label>First model<select data-compare-select="a">'+selectOptions(state.cmpA)+'</select></label><button class="button" data-swap aria-label="Swap selected models">Swap models</button><label>Second model<select data-compare-select="b">'+selectOptions(state.cmpB)+'</select></label></div><div class="empty">Choose two different models.</div>';}
    var diff=compareData(a,b);
    function sec(id,title){var x=diff.sections[id]||{rows:[],shared:0,diff:0};return sectionHTML(id,title,x.shared+' shared · '+x.diff+' different',x.rows);}
    return '<h2>Compare</h2><p class="intro">Decision summary first; expand technical evidence as needed.</p><div class="compare-controls"><label>First model<select data-compare-select="a">'+selectOptions(state.cmpA)+'</select></label><button class="button" data-swap aria-label="Swap selected models">Swap models</button><label>Second model<select data-compare-select="b">'+selectOptions(state.cmpB)+'</select></label><label><input type="checkbox" data-diff-only'+(state.cmpDiffOnly?' checked':'')+'> Show only differences</label></div><div class="compare-summary"><div class="identity-row">'+identity(state.cmpA)+'<div class="relation-word">compared with</div>'+identity(state.cmpB)+'</div><div class="summary-grid"><div class="summary-item"><span>Relationship</span><strong>'+esc(relationshipSummary(p))+'</strong></div><div class="summary-item"><span>Composite similarity</span><strong>'+score(p&&p.composite)+'</strong></div><div class="summary-item"><span>Containment direction</span><strong>'+esc(containmentSummary(p))+'</strong></div><div class="summary-item"><span>Object summary</span><strong>'+diff.shared+' shared · '+diff.differences+' different</strong></div></div></div>'+sec('tables','Tables')+sec('columns','Columns')+sec('measures','Measures and DAX')+sec('relationships','Relationships')+sec('datasources','Data sources');
  }

  function mapHTML(){
    if(!MLIST.length){return '<h2>Similarity map</h2><div class="empty">No comparable models are available.</div>';}
    var pmap=pairMap(),head='<tr><th></th>'+MLIST.map(function(m){var label=m.name+' — '+m.workspace;return '<th class="col" scope="col" title="'+esc(label)+'">'+esc(label)+'</th>';}).join('')+'</tr>';
    var rows=MLIST.map(function(r,i){var cells=MLIST.map(function(c,j){var rLabel=r.name+' — '+r.workspace,cLabel=c.name+' — '+c.workspace,pairLabel=rLabel+' and '+cLabel;if(i===j){return '<td><span class="matrix-cell diagonal" title="'+esc(rLabel)+': same model" aria-label="'+esc(rLabel)+': same model">—</span></td>';}var p=pmap[pairKey(r.id,c.id)];if(!p){return '<td><span class="matrix-cell unscored" title="'+esc(pairLabel)+': Not scored" aria-label="'+esc(pairLabel)+': Not scored"></span></td>';}var tier=tierAt(p.composite||0),cls=tier==='duplicate'?'duplicate':(tier==='similar'?'high':'low');return '<td><button class="matrix-cell '+cls+'" data-map-a="'+esc(r.id)+'" data-map-b="'+esc(c.id)+'" title="'+esc(pairLabel)+': '+score(p.composite)+'" aria-label="Compare '+esc(pairLabel)+', composite similarity '+score(p.composite)+'">'+Math.round((p.composite||0)*100)+'</button></td>';}).join('');return '<tr><th scope="row" title="'+esc(r.name)+' — '+esc(r.workspace)+'">'+esc(r.name)+'<div class="workspace">'+esc(r.workspace)+'</div></th>'+cells+'</tr>';}).join('');
    return '<h2>Similarity map</h2><p class="intro">All catalog models are listed. Select any scored cell to open Compare.</p><div class="map-note"><strong>Scope warning:</strong> when blocking is enabled, some model pairs are never scored. Blank cells are <strong>Not scored</strong>; they are not zero-similarity results.</div><div class="map-wrap"><table class="matrix" aria-label="Semantic model composite similarity map">'+head+rows+'</table><div class="legend" aria-label="Similarity map legend"><span><span class="legend-key duplicate"></span> Likely duplicate</span><span><span class="legend-key high"></span> High overlap</span><span><span class="legend-key low"></span> Low overlap / scored zero</span><span><span class="legend-key unscored"></span> Not scored (blank)</span></div></div>';
  }

  function viewHTML(){var content=state.tab==='review'?reviewHTML():(state.tab==='groups'?groupsHTML():(state.tab==='compare'?compareHTML():mapHTML()));return '<div class="view" role="tabpanel" id="panel-'+state.tab+'" aria-labelledby="tab-'+state.tab+'">'+content+'</div>';}
  function footHTML(){return '<div class="foot">Current thresholds: likely duplicate ≥ '+score(state.thresholds.duplicate)+' composite · high overlap ≥ '+score(state.thresholds.similar)+' composite · subset / superset ≥ '+score(state.thresholds.containment)+' directional coverage. Existing scores are reclassified only.</div>';}
  function render(){root.innerHTML=headerHTML()+settingsHTML()+tabsHTML()+viewHTML()+footHTML();applyTheme(false);wire();}
  function setTab(tab){state.tab=tab;render();}
  function openCompare(a,b,section){state.cmpA=String(a);state.cmpB=String(b);state.cmpDiffOnly=true;state.openSections={};if(section){state.openSections[section]=true;}setTab('compare');}
  function applyTheme(save){document.documentElement.setAttribute('data-theme',state.theme);if(save){try{localStorage.setItem('sms-theme',state.theme);}catch(e){}}}
  function applyThresholds(){state.thresholds=Object.assign({},state.draft);try{localStorage.setItem('sms-thresholds',JSON.stringify(state.thresholds));}catch(e){}render();}

  function wireCandidateActions(scope){
    scope.querySelectorAll('[data-evidence]').forEach(function(btn){btn.addEventListener('click',function(){var target=scope.querySelector('#'+btn.dataset.evidence),expanded=btn.getAttribute('aria-expanded')==='true';btn.setAttribute('aria-expanded',String(!expanded));if(target){target.hidden=expanded;}});});
    scope.querySelectorAll('[data-compare-a]').forEach(function(btn){btn.addEventListener('click',function(){openCompare(btn.dataset.compareA,btn.dataset.compareB,btn.dataset.openSection);});});
  }

  function wire(){
    root.querySelectorAll('[data-tab]').forEach(function(btn){btn.addEventListener('click',function(){setTab(btn.dataset.tab);});});
    var settings=root.querySelector('[data-settings]');if(settings){settings.addEventListener('click',function(){state.settingsOpen=!state.settingsOpen;render();});}
    root.querySelectorAll('[data-theme-set]').forEach(function(btn){btn.addEventListener('click',function(){state.theme=btn.dataset.themeSet;applyTheme(true);render();});});
    root.querySelectorAll('[data-draft-range],[data-draft-number]').forEach(function(input){input.addEventListener('input',function(){var key=input.dataset.draftRange||input.dataset.draftNumber,value=Math.max(0,Math.min(1,parseFloat(input.value)));if(isNaN(value)){return;}state.draft[key]=value;var other=root.querySelector(input.dataset.draftRange?'[data-draft-number="'+key+'"]':'[data-draft-range="'+key+'"]');if(other){other.value=input.dataset.draftRange?value.toFixed(2):value;}});});
    var apply=root.querySelector('[data-apply]');if(apply){apply.addEventListener('click',applyThresholds);}
    var reset=root.querySelector('[data-reset]');if(reset){reset.addEventListener('click',function(){state.draft=Object.assign({},DEFAULTS);applyThresholds();});}
    var search=root.querySelector('.search');if(search){search.addEventListener('input',function(){state.search=search.value;var queue=root.querySelector('.queue');if(queue){queue.innerHTML=reviewQueueHTML();wireCandidateActions(queue);}});}
    root.querySelectorAll('[data-filter]').forEach(function(btn){btn.addEventListener('click',function(){state.filter=btn.dataset.filter;render();});});
    wireCandidateActions(root);
    root.querySelectorAll('[data-group-side]').forEach(function(sel){sel.addEventListener('change',function(){var key=sel.dataset.group,side=sel.dataset.groupSide;state.groupSelections[key]=state.groupSelections[key]||{};state.groupSelections[key][side]=sel.value;});});
    root.querySelectorAll('[data-group-compare]').forEach(function(btn){btn.addEventListener('click',function(){var selected=state.groupSelections[btn.dataset.groupCompare];if(selected&&selected.a!==selected.b){openCompare(selected.a,selected.b,'tables');}});});
    root.querySelectorAll('[data-compare-select]').forEach(function(sel){sel.addEventListener('change',function(){if(sel.dataset.compareSelect==='a'){state.cmpA=sel.value;}else{state.cmpB=sel.value;}render();});});
    var swap=root.querySelector('[data-swap]');if(swap){swap.addEventListener('click',function(){var tmp=state.cmpA;state.cmpA=state.cmpB;state.cmpB=tmp;render();});}
    var only=root.querySelector('[data-diff-only]');if(only){only.addEventListener('change',function(){state.cmpDiffOnly=only.checked;render();});}
    root.querySelectorAll('[data-section]').forEach(function(btn){btn.addEventListener('click',function(){var id=btn.dataset.section;state.openSections[id]=!state.openSections[id];render();});});
    root.querySelectorAll('[data-map-a]').forEach(function(btn){btn.addEventListener('click',function(){openCompare(btn.dataset.mapA,btn.dataset.mapB,'tables');});});
  }

  render();
})();
</script>
"""

    app_json = json.dumps(app_data).replace("<", "\\u003c")
    displayHTML(app_template.replace("__APP_DATA__", app_json))

In [ ]:
render_results()